# AGMTL-DenseCBAM

Purpose: produce the experimental outputs needed for Chapter IV: system architecture, module-level traceability, sample system simulation, test results, statistical tests, graphs, Grad-CAM explainability, and paper-ready result tables.

This notebook supersedes `legacy/Main.ipynb` for thesis execution. The legacy notebook trains one fixed split and contains a disconnected attention branch; this notebook keeps the DenseNet201 benchmark controlled, connects attention into the forward path, adds the proposed CBAM + auxiliary artifact head, supports clean/artifact conditions, and exports Chapter IV artifacts.


## System architecture

```mermaid
flowchart TD
    A[Panoramic radiograph dataset] --> B[Dataset indexer]
    B --> C[Stratified 5-fold splitter]
    C --> D[Image loader and preprocessor]
    D --> E{Scenario}
    E -->|Clean| F[Normalized image]
    E -->|Artifact mix| G[Artifact injector: none / motion blur / Gaussian noise / metal streak]
    G --> F
    F --> H[DenseNet201 shared encoder]
    H --> I1[Benchmark self-attention fusion]
    H --> I2[Proposed CBAM spatial-channel attention]
    I1 --> J1[TMD classifier head]
    I2 --> J2[TMD classifier head]
    I2 --> J3[Artifact classifier auxiliary head]
    J1 --> K[Predictions: Normal / Subluxation]
    J2 --> K
    J3 --> L[Artifact prediction]
    K --> M[Confusion matrix and metrics]
    K --> N[Grad-CAM and localization energy]
    M --> O[Mean ± SD, Shapiro-Wilk, paired t-test/Wilcoxon]
    N --> P[Explainability figures]
    O --> Q[Chapter IV tables and plots]
    P --> Q
```


In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import random
import shutil
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy import stats
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold, train_test_split
from tensorflow.keras import Model, layers, mixed_precision
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import Sequence as KerasSequence, to_categorical

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "1")
tf.config.optimizer.set_jit(False)
try:
    mixed_precision.set_global_policy("mixed_float16")
except Exception as exc:
    print(f"Mixed precision not enabled: {exc}")

In [ ]:
TMD_LABELS = ["normal", "subluxation"]
DISPLAY_LABELS = ["Normal", "Subluxation"]
ARTIFACT_LABELS = ["none", "motion_blur", "gaussian_noise", "metal_streak"]
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

@dataclass
class ExperimentConfig:
    dataset_root: str
    image_size: Tuple[int, int] = (224, 224)
    batch_size: int = 8
    epochs: int = 50
    learning_rate: float = 1e-4
    weight_decay: float = 1e-2
    random_state: int = 42
    freeze_backbone: bool = False
    tmd_loss_weight: float = 1.0
    artifact_loss_weight: float = 0.35

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

In [ ]:
def index_dataset(dataset_root: str) -> pd.DataFrame:
    root = Path(dataset_root)
    if not root.exists():
        raise FileNotFoundError(f"Dataset root does not exist: {dataset_root}")
    rows = []
    class_to_idx = {name: idx for idx, name in enumerate(TMD_LABELS)}
    for split_name in ["train", "validation", "test"]:
        split_dir = root / split_name
        if not split_dir.exists():
            continue
        for class_name in TMD_LABELS:
            class_dir = split_dir / class_name
            if not class_dir.exists():
                raise FileNotFoundError(f"Missing class folder: {class_dir}")
            for file_path in class_dir.rglob("*"):
                if file_path.is_file() and file_path.suffix.lower() in IMAGE_EXTENSIONS:
                    rows.append({"filepath": str(file_path), "tmd_label": class_to_idx[class_name], "class_name": class_name, "split": split_name})
    if not rows:
        for class_name in TMD_LABELS:
            class_dir = root / class_name
            if not class_dir.exists():
                raise FileNotFoundError(f"Expected {class_dir} for flat-layout CV input")
            for file_path in class_dir.rglob("*"):
                if file_path.is_file() and file_path.suffix.lower() in IMAGE_EXTENSIONS:
                    rows.append({"filepath": str(file_path), "tmd_label": class_to_idx[class_name], "class_name": class_name, "split": "all"})
    if not rows:
        raise ValueError("No image files found.")
    return pd.DataFrame(rows).sort_values("filepath").reset_index(drop=True)

def summarize_dataset(dataset_root: str) -> pd.DataFrame:
    df = index_dataset(dataset_root)
    return df.groupby(["split", "class_name"]).size().reset_index(name="count")

In [ ]:
def _ensure_uint8(image: np.ndarray) -> np.ndarray:
    return np.clip(image, 0, 255).astype(np.uint8)

def add_motion_blur(image: np.ndarray, kernel_size: int = 9) -> np.ndarray:
    kernel_size = max(3, int(kernel_size) | 1)
    kernel = np.zeros((kernel_size, kernel_size), dtype=np.float32)
    kernel[kernel_size // 2, :] = 1.0 / kernel_size
    return _ensure_uint8(cv2.filter2D(image, -1, kernel))

def add_gaussian_noise(image: np.ndarray, sigma: float = 18.0) -> np.ndarray:
    return _ensure_uint8(image.astype(np.float32) + np.random.normal(0, sigma, image.shape))

def add_metal_streak(image: np.ndarray, num_streaks: int = 3) -> np.ndarray:
    h, w = image.shape[:2]
    output = image.copy().astype(np.float32)
    for _ in range(max(1, num_streaks)):
        overlay = np.zeros_like(output)
        cv2.line(overlay, (np.random.randint(0, w), np.random.randint(0, h)), (np.random.randint(0, w), np.random.randint(0, h)), (np.random.randint(180, 255),) * 3, np.random.randint(2, 8))
        overlay = cv2.GaussianBlur(overlay, (0, 0), sigmaX=2.0)
        output = np.maximum(output, overlay)
    return _ensure_uint8(output)

def apply_artifact(image: np.ndarray, artifact_label: int) -> np.ndarray:
    return [lambda x: x, add_motion_blur, add_gaussian_noise, add_metal_streak][artifact_label](image)

In [ ]:
class TMJSequence(KerasSequence):
    def __init__(self, dataframe: pd.DataFrame, image_size: Tuple[int, int], batch_size: int, multi_task: bool, scenario: str, training: bool, seed: int = 42):
        self.df = dataframe.reset_index(drop=True).copy()
        self.image_size = image_size
        self.batch_size = batch_size
        self.multi_task = multi_task
        self.scenario = scenario
        self.training = training
        self.rng = np.random.default_rng(seed)
        self.indices = np.arange(len(self.df))
        self.on_epoch_end()
    def __len__(self):
        return math.ceil(len(self.df) / self.batch_size)
    def on_epoch_end(self):
        if self.training:
            self.rng.shuffle(self.indices)
    def _artifact_for(self, filepath: str) -> int:
        if self.scenario == "clean":
            return 0
        if self.training:
            return int(self.rng.integers(0, len(ARTIFACT_LABELS)))
        return int(hashlib.md5(filepath.encode("utf-8")).hexdigest(), 16) % len(ARTIFACT_LABELS)
    def _load_image(self, filepath: str) -> np.ndarray:
        image = cv2.imread(filepath)
        if image is None:
            raise ValueError(f"Failed to load image: {filepath}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        return cv2.resize(image, self.image_size, interpolation=cv2.INTER_AREA)
    def __getitem__(self, index: int):
        batch_df = self.df.iloc[self.indices[index * self.batch_size:(index + 1) * self.batch_size]]
        images, tmd_labels, artifact_labels = [], [], []
        for row in batch_df.itertuples(index=False):
            image = self._load_image(row.filepath)
            if self.training and self.rng.random() < 0.5:
                image = cv2.flip(image, 1)
            artifact_label = self._artifact_for(row.filepath)
            image = apply_artifact(image, artifact_label).astype(np.float32) / 255.0
            images.append(image); tmd_labels.append(int(row.tmd_label)); artifact_labels.append(artifact_label)
        x = np.stack(images)
        y_tmd = to_categorical(np.array(tmd_labels), num_classes=len(TMD_LABELS))
        y_artifact = to_categorical(np.array(artifact_labels), num_classes=len(ARTIFACT_LABELS))
        return (x, {"tmd_output": y_tmd, "artifact_output": y_artifact}) if self.multi_task else (x, y_tmd)

In [ ]:
class AttentionBlock(layers.Layer):
    def __init__(self, attention_type: str = "cbam", reduction_ratio: int = 16, **kwargs):
        super().__init__(**kwargs)
        self.attention_type = attention_type.lower(); self.reduction_ratio = reduction_ratio
    def build(self, input_shape):
        filters = int(input_shape[-1]); reduced = max(filters // self.reduction_ratio, 1)
        if self.attention_type == "self":
            qk = max(filters // 8, 1)
            self.query_conv = layers.Conv2D(qk, 1, padding="same")
            self.key_conv = layers.Conv2D(qk, 1, padding="same")
            self.value_conv = layers.Conv2D(filters, 1, padding="same")
        elif self.attention_type == "cbam":
            self.avg_pool = layers.GlobalAveragePooling2D(); self.max_pool = layers.GlobalMaxPooling2D()
            self.shared_dense_1 = layers.Dense(reduced, activation="relu")
            self.shared_dense_2 = layers.Dense(filters)
            self.spatial_conv = layers.Conv2D(1, 7, padding="same", activation="sigmoid")
        else:
            raise ValueError(f"Unsupported attention type: {self.attention_type}")
        super().build(input_shape)
    def call(self, inputs):
        if self.attention_type == "self":
            shp = tf.shape(inputs); b, h, w = shp[0], shp[1], shp[2]; c = inputs.shape[-1]
            q = tf.reshape(self.query_conv(inputs), [b, h*w, -1])
            k = tf.reshape(self.key_conv(inputs), [b, h*w, -1])
            v = tf.reshape(self.value_conv(inputs), [b, h*w, c])
            att = tf.nn.softmax(tf.matmul(q, k, transpose_b=True) / tf.math.sqrt(tf.cast(tf.shape(k)[-1], tf.float32)), axis=-1)
            return inputs + tf.reshape(tf.matmul(att, v), [b, h, w, c])
        avg = self.shared_dense_2(self.shared_dense_1(self.avg_pool(inputs)))
        mx = self.shared_dense_2(self.shared_dense_1(self.max_pool(inputs)))
        ca = tf.reshape(tf.nn.sigmoid(avg + mx), (-1, 1, 1, inputs.shape[-1]))
        x = inputs * ca
        sa = self.spatial_conv(tf.concat([tf.reduce_mean(x, axis=-1, keepdims=True), tf.reduce_max(x, axis=-1, keepdims=True)], axis=-1))
        return x * sa

def _make_backbone(config: ExperimentConfig) -> Model:
    backbone = tf.keras.applications.DenseNet201(include_top=False, weights="imagenet", input_shape=(*config.image_size, 3), pooling=None)
    backbone.trainable = not config.freeze_backbone
    return backbone

def build_benchmark_model(config: ExperimentConfig) -> Model:
    backbone = _make_backbone(config)
    pool3 = backbone.get_layer("pool3_relu").output
    pool3_att = AttentionBlock("self", name="benchmark_self_attention")(pool3)
    pool3_down = layers.AveragePooling2D(pool_size=2, name="benchmark_pool3_downsample")(pool3_att)
    conv5 = backbone.get_layer("conv5_block32_concat").output
    pool3_proj = layers.Conv2D(int(conv5.shape[-1]), 1, padding="same", name="benchmark_pool3_projection")(pool3_down)
    fused = layers.Concatenate(name="benchmark_fused_features")([conv5, pool3_proj])
    fused = layers.Conv2D(1024, 1, activation="relu", padding="same", name="benchmark_fusion_conv")(fused)
    gap = layers.GlobalAveragePooling2D(name="benchmark_gap")(fused)
    x = layers.Dense(512, activation="relu", kernel_regularizer=l2(config.weight_decay), name="benchmark_fc1")(gap)
    x = layers.Dropout(0.5, name="benchmark_dropout")(x)
    x = layers.BatchNormalization(name="benchmark_bn")(x)
    x = layers.Dense(128, activation="relu", name="benchmark_fc2")(x)
    out = layers.Dense(len(TMD_LABELS), activation="softmax", dtype="float32", name="tmd_output")(x)
    return Model(backbone.input, out, name="DenseNet201_Benchmark_SelfAttention")

def build_agmtl_densecbam_model(config: ExperimentConfig) -> Model:
    backbone = _make_backbone(config)
    conv5 = backbone.get_layer("conv5_block32_concat").output
    attended = AttentionBlock("cbam", name="cbam_attention")(conv5)
    refined = layers.Conv2D(1024, 1, activation="relu", padding="same", name="cbam_refine_conv")(attended)
    shared = layers.GlobalAveragePooling2D(name="shared_gap")(refined)
    shared = layers.Dense(512, activation="relu", kernel_regularizer=l2(config.weight_decay), name="shared_fc1")(shared)
    shared = layers.Dropout(0.5, name="shared_dropout")(shared)
    shared = layers.BatchNormalization(name="shared_bn")(shared)
    shared = layers.Dense(128, activation="relu", name="shared_fc2")(shared)
    tmd = layers.Dense(len(TMD_LABELS), activation="softmax", dtype="float32", name="tmd_output")(shared)
    artifact = layers.Dense(len(ARTIFACT_LABELS), activation="softmax", dtype="float32", name="artifact_output")(shared)
    return Model(backbone.input, [tmd, artifact], name="AGMTL_DenseCBAM")

In [ ]:
def compile_model(model: Model, multi_task: bool, config: ExperimentConfig) -> None:
    if multi_task:
        model.compile(optimizer=Adam(config.learning_rate), loss={"tmd_output":"categorical_crossentropy", "artifact_output":"categorical_crossentropy"}, loss_weights={"tmd_output":config.tmd_loss_weight, "artifact_output":config.artifact_loss_weight}, metrics={"tmd_output":["accuracy"], "artifact_output":["accuracy"]}, jit_compile=False)
    else:
        model.compile(optimizer=Adam(config.learning_rate), loss="categorical_crossentropy", metrics=["accuracy"], jit_compile=False)

def make_callbacks(output_dir: Path, fold_name: str, model_type: str, scenario: str):
    output_dir.mkdir(parents=True, exist_ok=True)
    return [
        ModelCheckpoint(str(output_dir / f"{fold_name}_{model_type}_{scenario}_best.keras"), monitor="val_loss", save_best_only=True, verbose=1),
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-6, verbose=1),
    ]

def compute_binary_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    return {"accuracy": float((tp+tn)/max(cm.sum(),1)), "precision": float(precision), "recall": float(recall), "specificity": float(tn/max(tn+fp,1)), "f1": float(f1), "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

def collect_predictions(model: Model, generator: TMJSequence, multi_task: bool):
    y_true, y_pred, y_prob = [], [], []
    for i in range(len(generator)):
        x, y = generator[i]
        pred = model.predict(x, verbose=0)
        pred_tmd = pred[0] if multi_task else pred
        true_tmd = y["tmd_output"] if multi_task else y
        y_true.extend(np.argmax(true_tmd, axis=1)); y_pred.extend(np.argmax(pred_tmd, axis=1)); y_prob.extend(np.max(pred_tmd, axis=1))
    return np.array(y_true), np.array(y_pred), np.array(y_prob)

def measure_inference_speed(model: Model, generator: TMJSequence) -> Tuple[float, float]:
    if len(generator) == 0:
        raise ValueError("Empty generator")
    x0, _ = generator[0]; model.predict(x0, verbose=0)
    total = 0; start = time.perf_counter()
    for i in range(len(generator)):
        x, _ = generator[i]; model.predict(x, verbose=0); total += len(x)
    elapsed = time.perf_counter() - start
    ips = total / max(elapsed, 1e-8)
    return ips, 1000.0 / max(ips, 1e-8)

In [ ]:
def run_fixed_split_experiment(config: ExperimentConfig, model_type: str, scenario: str, output_dir: Path) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if model_type not in {"benchmark", "proposed"}: raise ValueError("model_type must be benchmark or proposed")
    if scenario not in {"clean", "artifact_mix"}: raise ValueError("scenario must be clean or artifact_mix")
    seed_everything(config.random_state)
    df = index_dataset(config.dataset_root)
    train_df = df[df.split == "train"].reset_index(drop=True); val_df = df[df.split == "validation"].reset_index(drop=True); test_df = df[df.split == "test"].reset_index(drop=True)
    if min(len(train_df), len(val_df), len(test_df)) == 0: raise ValueError("Fixed-split run requires train/validation/test folders")
    multi_task = model_type == "proposed"
    train_gen = TMJSequence(train_df, config.image_size, config.batch_size, multi_task, scenario, True, config.random_state)
    val_gen = TMJSequence(val_df, config.image_size, config.batch_size, multi_task, scenario, False, config.random_state)
    test_gen = TMJSequence(test_df, config.image_size, config.batch_size, multi_task, scenario, False, config.random_state)
    model = build_agmtl_densecbam_model(config) if multi_task else build_benchmark_model(config)
    compile_model(model, multi_task, config)
    history = model.fit(train_gen, validation_data=val_gen, epochs=config.epochs, callbacks=make_callbacks(output_dir, Path(config.dataset_root).name, model_type, scenario), verbose=1)
    y_true, y_pred, y_prob = collect_predictions(model, test_gen, multi_task)
    metrics = compute_binary_metrics(y_true, y_pred)
    ips, latency_ms = measure_inference_speed(model, test_gen)
    result = pd.DataFrame([{ "model_type": model_type, "scenario": scenario, "fold": Path(config.dataset_root).name, **metrics, "images_per_second": ips, "latency_ms": latency_ms, "epochs_ran": len(history.history.get("loss", [])) }])
    cm = pd.DataFrame(confusion_matrix(y_true, y_pred, labels=[0,1]), index=DISPLAY_LABELS, columns=DISPLAY_LABELS)
    pred_df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred, "confidence": y_prob})
    return result, cm, pred_df

def run_all_cases(folds_root: Path, output_root: Path, cases: Sequence[Tuple[str, str]], image_size=(224,224), batch_size=8, epochs=50, learning_rate=1e-4):
    fold_dirs = sorted([p for p in folds_root.iterdir() if p.is_dir() and p.name.startswith("fold_")], key=lambda p: int(p.name.split("_")[1]))
    all_rows = []
    for model_type, scenario in cases:
        case_dir = output_root / f"{model_type}_{scenario}"
        cms = []
        for fold_dir in fold_dirs:
            print(f"Running {model_type} / {scenario} / {fold_dir.name}")
            cfg = ExperimentConfig(str(fold_dir), image_size=image_size, batch_size=batch_size, epochs=epochs, learning_rate=learning_rate)
            result, cm, pred = run_fixed_split_experiment(cfg, model_type, scenario, case_dir)
            result.to_csv(case_dir / f"{fold_dir.name}_results.csv", index=False)
            cm.to_csv(case_dir / f"{fold_dir.name}_confusion_matrix.csv")
            pred.to_csv(case_dir / f"{fold_dir.name}_predictions.csv", index=False)
            cms.append(cm); all_rows.extend(result.to_dict("records"))
            tf.keras.backend.clear_session(); gc.collect()
        pooled = cms[0].copy()
        for cm in cms[1:]: pooled = pooled.add(cm, fill_value=0)
        pooled.astype(int).to_csv(case_dir / "pooled_confusion_matrix.csv")
    combined = pd.DataFrame(all_rows)
    output_root.mkdir(parents=True, exist_ok=True)
    combined.to_csv(output_root / "all_case_fold_results.csv", index=False)
    return combined

## Run configuration

Use the four thesis cases for final Chapter IV evidence:

- C1: benchmark + clean
- C2: benchmark + artifact_mix
- C3: proposed + clean
- C4: proposed + artifact_mix

Start with `EPOCHS = 2` for a smoke test, then use `EPOCHS = 50` for final runs.


In [ ]:
FOLDS_ROOT = Path("data_5_fold")
OUTPUT_ROOT = Path("chapter4_results")
CASES = [("benchmark", "clean"), ("benchmark", "artifact_mix"), ("proposed", "clean"), ("proposed", "artifact_mix")]
BATCH_SIZE = 8
EPOCHS = 50
IMAGE_SIZE = (224, 224)
LEARNING_RATE = 1e-4

# Uncomment for final execution after data_5_fold exists:
# all_case_results = run_all_cases(FOLDS_ROOT, OUTPUT_ROOT, CASES, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, epochs=EPOCHS, learning_rate=LEARNING_RATE)
# all_case_results

In [ ]:
def load_existing_results(result_root: Path) -> pd.DataFrame:
    rows = []
    for csv_path in result_root.rglob("fold_*_results.csv"):
        rows.append(pd.read_csv(csv_path))
    if not rows:
        raise FileNotFoundError(f"No fold result CSVs found under {result_root}")
    return pd.concat(rows, ignore_index=True)

def summarize_results(results_df: pd.DataFrame) -> pd.DataFrame:
    metric_cols = ["accuracy", "precision", "recall", "specificity", "f1", "images_per_second", "latency_ms", "epochs_ran"]
    existing = [c for c in metric_cols if c in results_df.columns]
    grouped = results_df.groupby(["model_type", "scenario"])[existing].agg(["mean", "std", "count"])
    grouped.columns = ["_".join(col).strip() for col in grouped.columns]
    return grouped.reset_index()

def chapter_table(results_df: pd.DataFrame) -> pd.DataFrame:
    summary = summarize_results(results_df)
    rows = []
    for _, r in summary.iterrows():
        row = {"Model": r["model_type"], "Data condition": r["scenario"]}
        for metric in ["accuracy", "precision", "recall", "specificity", "f1", "images_per_second"]:
            mean_col, std_col = f"{metric}_mean", f"{metric}_std"
            if mean_col in summary.columns:
                row[metric] = f"{r[mean_col]:.4f} ± {r[std_col]:.4f}" if not pd.isna(r[std_col]) else f"{r[mean_col]:.4f}"
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
def paired_statistical_tests(results_df: pd.DataFrame, metric: str = "f1") -> pd.DataFrame:
    rows = []
    for scenario in sorted(results_df["scenario"].unique()):
        bench = results_df[(results_df.model_type == "benchmark") & (results_df.scenario == scenario)].sort_values("fold")
        prop = results_df[(results_df.model_type == "proposed") & (results_df.scenario == scenario)].sort_values("fold")
        merged = bench[["fold", metric]].merge(prop[["fold", metric]], on="fold", suffixes=("_benchmark", "_proposed"))
        if len(merged) < 2:
            continue
        diff = merged[f"{metric}_proposed"] - merged[f"{metric}_benchmark"]
        shapiro_stat, shapiro_p = stats.shapiro(diff) if len(diff) >= 3 else (np.nan, np.nan)
        if pd.notna(shapiro_p) and shapiro_p > 0.05:
            test_name = "paired_t_test"; stat, p = stats.ttest_rel(merged[f"{metric}_proposed"], merged[f"{metric}_benchmark"])
        else:
            test_name = "wilcoxon_signed_rank"; stat, p = stats.wilcoxon(merged[f"{metric}_proposed"], merged[f"{metric}_benchmark"], zero_method="wilcox", alternative="two-sided")
        rows.append({"scenario": scenario, "metric": metric, "n_pairs": len(merged), "mean_difference": float(diff.mean()), "shapiro_w": shapiro_stat, "shapiro_p": shapiro_p, "selected_test": test_name, "test_statistic": stat, "p_value": p})
    return pd.DataFrame(rows)

def expected_calibration_error(y_true: np.ndarray, y_pred: np.ndarray, confidence: np.ndarray, n_bins: int = 10) -> float:
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (confidence > lo) & (confidence <= hi)
        if not np.any(mask):
            continue
        acc = np.mean(y_true[mask] == y_pred[mask])
        conf = np.mean(confidence[mask])
        ece += np.mean(mask) * abs(acc - conf)
    return float(ece)

In [ ]:
def plot_metric_bars(results_df: pd.DataFrame, output_dir: Path = Path("chapter4_results/figures")) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    table = results_df.melt(id_vars=["model_type", "scenario", "fold"], value_vars=["accuracy", "precision", "recall", "specificity", "f1"], var_name="metric", value_name="value")
    for metric, part in table.groupby("metric"):
        plt.figure(figsize=(8, 5))
        labels = [f"{m}\n{s}" for m, s in part.groupby(["model_type", "scenario"]).groups.keys()]
        means = part.groupby(["model_type", "scenario"])["value"].mean().values
        stds = part.groupby(["model_type", "scenario"])["value"].std().values
        plt.bar(labels, means, yerr=stds, capsize=4)
        plt.ylim(0, 1)
        plt.ylabel(metric)
        plt.title(f"{metric.title()} by Model and Data Condition")
        plt.tight_layout()
        plt.savefig(output_dir / f"{metric}_bar.png", dpi=300)
        plt.show()

def plot_fold_lines(results_df: pd.DataFrame, metric: str = "f1", output_dir: Path = Path("chapter4_results/figures")) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(8,5))
    for (model_type, scenario), part in results_df.groupby(["model_type", "scenario"]):
        part = part.sort_values("fold")
        plt.plot(part["fold"], part[metric], marker="o", label=f"{model_type}-{scenario}")
    plt.ylim(0, 1); plt.ylabel(metric); plt.xlabel("Fold"); plt.title(f"{metric.title()} across 5 folds"); plt.legend(); plt.tight_layout()
    plt.savefig(output_dir / f"{metric}_fold_line.png", dpi=300)
    plt.show()

In [ ]:
def find_target_layer(model: Model, preferred: Optional[str] = None) -> str:
    if preferred: return preferred
    for name in ["cbam_refine_conv", "benchmark_fusion_conv", "conv5_block32_concat"]:
        try:
            model.get_layer(name); return name
        except ValueError:
            pass
    raise ValueError("No Grad-CAM layer found")

def make_gradcam_heatmap(model: Model, image_array: np.ndarray, target_class: Optional[int] = None, target_layer: Optional[str] = None) -> np.ndarray:
    target_layer = find_target_layer(model, target_layer)
    image_batch = image_array[None, ...] if image_array.ndim == 3 else image_array
    grad_model = Model(model.inputs, [model.get_layer(target_layer).output, model.get_layer("tmd_output").output])
    with tf.GradientTape() as tape:
        conv_outputs, preds = grad_model(image_batch)
        if target_class is None: target_class = int(tf.argmax(preds[0]))
        class_channel = preds[:, target_class]
    grads = tape.gradient(class_channel, conv_outputs)
    weights = tf.reduce_mean(grads, axis=(0,1,2))
    heatmap = tf.reduce_sum(conv_outputs[0] * weights, axis=-1)
    return (tf.maximum(heatmap, 0) / tf.maximum(tf.reduce_max(heatmap), 1e-8)).numpy()

def localization_energy(heatmap: np.ndarray, roi_mask: np.ndarray) -> float:
    resized = cv2.resize(heatmap, (roi_mask.shape[1], roi_mask.shape[0]))
    roi = (roi_mask > 0).astype(np.float32)
    return float((resized * roi).sum() / max(resized.sum(), 1e-8))

## Sample system simulation of test data

Run the next cell after setting `SAMPLE_IMAGE_PATH`. It exports a clean image, motion-blur image, Gaussian-noise image, metal-streak image, and a file dump of the tensor shape and labels. Use these outputs in Chapter IV's sample simulation subsection.


In [ ]:
def simulate_sample_image(sample_image_path: str, output_dir: Path = Path("chapter4_results/sample_simulation")) -> pd.DataFrame:
    output_dir.mkdir(parents=True, exist_ok=True)
    image = cv2.imread(sample_image_path)
    if image is None: raise ValueError(f"Cannot load {sample_image_path}")
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (224,224), interpolation=cv2.INTER_AREA)
    records = []
    for idx, label in enumerate(ARTIFACT_LABELS):
        transformed = apply_artifact(image, idx)
        out_path = output_dir / f"sample_{label}.png"
        cv2.imwrite(str(out_path), cv2.cvtColor(transformed, cv2.COLOR_RGB2BGR))
        tensor = transformed.astype(np.float32) / 255.0
        records.append({"stage": label, "output_file": str(out_path), "shape": str(tensor.shape), "min": float(tensor.min()), "max": float(tensor.max()), "mean": float(tensor.mean())})
    df = pd.DataFrame(records)
    df.to_csv(output_dir / "sample_file_dump.csv", index=False)
    return df

# SAMPLE_IMAGE_PATH = "data_5_fold/fold_1/test/normal/example.png"
# simulate_sample_image(SAMPLE_IMAGE_PATH)